# 🥉 Notebook 01 — Bronze Ingestion
**Layer**: Bronze — raw ingestion (append-only, immutable)
**Runtime**: ~5–8 min for 10K members / 24 months

## What this notebook does
1. Creates Unity Catalog infrastructure (Catalog → Schema → Volume)
2. Generates 10,000 synthetic Medicare members + ~1M claim lines in pandas
3. Persists raw CSVs to Unity Catalog Volume (governed landing zone)
4. Reads CSVs into Spark with declared `StructType` schema enforcement
5. Adds audit columns (`_ingested_at`, `_pipeline_layer`, `_batch_id`, `_bronze_null_flag`)
6. Writes to Bronze Delta tables partitioned by `claim_year/claim_month`
7. Runs `OPTIMIZE + ZORDER BY bene_id` for downstream Silver query performance

**Serverless note**: No `spark.sparkContext` used anywhere — fully compatible.
**Design principle**: Bronze is append-only and immutable — the audit trail for HIPAA compliance.


## 0. Dependencies

In [0]:
%pip install xgboost==1.7.6 shap==0.43.0 scikit-learn==1.3.0
dbutils.library.restartPython()


## 1. Configuration

In [0]:
dbutils.widgets.text("n_members", "10000", "Number of synthetic members")
dbutils.widgets.text("n_months",  "24",    "Months of claims history")
dbutils.widgets.text("batch_id",  "",      "Batch ID (blank = auto-timestamp)")

N_MEMBERS = int(dbutils.widgets.get("n_members"))
N_MONTHS  = int(dbutils.widgets.get("n_months"))
BATCH_ID  = dbutils.widgets.get("batch_id") or None

print(f"N_MEMBERS : {N_MEMBERS:,}")
print(f"N_MONTHS  : {N_MONTHS}")
print(f"BATCH_ID  : {BATCH_ID or 'auto'}")


## 2. Repo path + catalog setup

In [0]:
import sys, os

# Auto-detect repo root — works for any user / workspace path
_nb_path  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/Workspace" + "/".join(_nb_path.split("/")[:-2])

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

CATALOG       = "medicare_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
ML_SCHEMA     = "ml_features"
VOLUME_PATH   = f"/Volumes/{CATALOG}/default/landing_zone/raw"

print(f"Repo root  : {repo_root}")
print(f"Catalog    : {CATALOG}")
print(f"Volume path: {VOLUME_PATH}")
_ok = os.path.exists(os.path.join(repo_root, "src"))
print("src/ found : ✅" if _ok else "❌ src/ missing — check repo clone")


## 3. Unity Catalog infrastructure
Creates **Catalog → Schemas → Volume** — all idempotent (`IF NOT EXISTS`).
The Volume provides HIPAA-lineage-tracked file storage at the landing zone.


In [0]:
# Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

# Schemas
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, ML_SCHEMA, "default"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    print(f"  Schema: {CATALOG}.{schema} ✅")

# Unity Catalog Volume for raw file landing
create_vol_sql = (
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.default.landing_zone "
    f"COMMENT 'CMS claims landing zone — append-only, HIPAA deidentified'"
)
spark.sql(create_vol_sql)
print(f"  Volume: {CATALOG}.default.landing_zone ✅")
print(f"  Path  : {VOLUME_PATH}")


## 4. Generate synthetic CMS claims
Runs on the driver in pandas — no Spark workers needed for data generation.

**Generator characteristics**:
- 27-condition ICD-10 catalogue with prevalence-weighted sampling
- Age-adjusted condition loading (Poisson-distributed encounter counts)
- Realistic service type mix: IP 8%, ED 12%, specialist 25%, primary 40%, lab 15%
- Intervention/control arm assignment with pre-specified cost reduction
- Deterministic: same seed = identical data every run (pipeline reproducibility)


In [0]:
from src.data_generator.cms_claims_generator import generate_and_save

members_df, claims_df = generate_and_save(
    output_dir          = VOLUME_PATH,
    n_members           = N_MEMBERS,
    n_months            = N_MONTHS,
    intervention_effect = -420.0,   # PMPM cost reduction for treated arm
    seed                = 42,
)

print(f"\n✅ Generated {len(members_df):,} members | {len(claims_df):,} claim lines")
print(f"   Saved to: {VOLUME_PATH}")
print(f"\nIntervention arm split:")
print(members_df["intervention_arm"].value_counts().to_string())
print(f"\nService type distribution:")
print(claims_df["service_type"].value_counts().to_string())
display(spark.createDataFrame(members_df.head(5)))


### Claims sample

In [0]:
display(spark.createDataFrame(claims_df.head(10)))


## 5. Bronze ingestion — schema enforcement + audit columns

### Why inline Spark (not `run_bronze_ingestion()`)?
On Databricks Serverless, the `src/bronze/ingest_claims.py` module calls `.count()` multiple
times during validation — each call is a separate Spark Connect round-trip, and chaining
them inside a class method triggers the 80-second serverless timeout. The correct pattern
is to perform all Spark transformations as a **single lazy DAG** and trigger exactly
one action (the write). This notebook does that.

### Production concerns implemented
- **Schema enforcement**: declared `StructType` — malformed files fail at the gate
- **Idempotency**: `_batch_id` prevents duplicate ingestion on pipeline re-run
- **Audit columns**: `_ingested_at`, `_pipeline_layer`, `_batch_id` on every row
- **Null flagging**: `_bronze_null_flag` marks critical null rows for Silver exclusion
- **Partition + ZORDER**: `claim_year/claim_month` + `ZORDER BY bene_id`


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from datetime import datetime

BATCH_ID = BATCH_ID or datetime.utcnow().strftime("%Y%m%d%H%M%S")
print(f"Batch ID: {BATCH_ID}")

# ── Declared StructType schemas — schema violations caught at read time ──────
CLAIMS_SCHEMA = StructType([
    StructField("claim_id",           StringType(),  False),
    StructField("bene_id",            StringType(),  False),
    StructField("service_date",       StringType(),  True),
    StructField("claim_year",         IntegerType(), True),
    StructField("claim_month",        IntegerType(), True),
    StructField("service_type",       StringType(),  True),
    StructField("icd10_primary",      StringType(),  True),
    StructField("icd10_codes",        StringType(),  True),
    StructField("provider_specialty", StringType(),  True),
    StructField("claim_amount",       DoubleType(),  True),
    StructField("allowed_amount",     DoubleType(),  True),
    StructField("paid_amount",        DoubleType(),  True),
    StructField("plan_type",          StringType(),  True),
    StructField("intervention_arm",   IntegerType(), True),
])

MEMBERS_SCHEMA = StructType([
    StructField("bene_id",          StringType(),  False),
    StructField("age",              IntegerType(), True),
    StructField("sex",              StringType(),  True),
    StructField("dual_eligible",    IntegerType(), True),
    StructField("plan_type",        StringType(),  True),
    StructField("state",            StringType(),  True),
    StructField("enrollment_date",  StringType(),  True),
    StructField("intervention_arm", IntegerType(), True),
])
print("✅ Schemas declared")


In [0]:
# ── Read CSVs with schema enforcement ────────────────────────────────────────
claims_raw = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(CLAIMS_SCHEMA)
    .csv(f"{VOLUME_PATH}/claims.csv")
)

members_raw = (
    spark.read
    .option("header", "true")
    .schema(MEMBERS_SCHEMA)
    .csv(f"{VOLUME_PATH}/members.csv")
)

# ── Build full Bronze transformation as a SINGLE lazy DAG ────────────────────
# One trigger (the write) — avoids Serverless Spark Connect timeout
claims_bronze = (
    claims_raw
    .withColumn("_ingested_at",      F.current_timestamp())
    .withColumn("_pipeline_layer",   F.lit("bronze"))
    .withColumn("_batch_id",         F.lit(BATCH_ID))
    .withColumn("_bronze_null_flag",
        F.col("claim_id").isNull() | F.col("bene_id").isNull()
    )
)

members_bronze = (
    members_raw
    .withColumn("_ingested_at",      F.current_timestamp())
    .withColumn("_pipeline_layer",   F.lit("bronze"))
    .withColumn("_batch_id",         F.lit(BATCH_ID))
    .withColumn("_bronze_null_flag", F.col("bene_id").isNull())
)

# ── Write claims — one action, partitioned + ZORDER ──────────────────────────
print("Writing bronze.raw_claims...")
(
    claims_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("claim_year", "claim_month")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.raw_claims")
)
spark.sql(f"OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.raw_claims ZORDER BY (bene_id)")
print(f"✅ {CATALOG}.{BRONZE_SCHEMA}.raw_claims — partitioned year/month, ZORDER bene_id")

# ── Write members ─────────────────────────────────────────────────────────────
print("Writing bronze.raw_members...")
(
    members_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.raw_members")
)
print(f"✅ {CATALOG}.{BRONZE_SCHEMA}.raw_members")


## 6. Verification

In [0]:
bronze_claims  = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.raw_claims")
bronze_members = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.raw_members")

print(f"bronze.raw_claims  rows: {bronze_claims.count():,}")
print(f"bronze.raw_members rows: {bronze_members.count():,}")
print()

# Partition distribution — shows data spread across time
print("=== Claim volume by year/month ===")
display(
    bronze_claims
    .groupBy("claim_year", "claim_month")
    .agg(
        F.count("*").alias("n_claims"),
        F.countDistinct("bene_id").alias("n_members"),
        F.round(F.sum("claim_amount"), 0).alias("total_cost_usd"),
        F.round(F.avg("claim_amount"), 2).alias("avg_claim_usd"),
    )
    .orderBy("claim_year", "claim_month")
)


In [0]:
# Null check — expect 0 on all critical columns
null_summary = bronze_claims.agg(*[
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["claim_id", "bene_id", "service_date", "claim_amount", "icd10_primary"]
])
print("=== Null counts (expect 0 on all) ===")
display(null_summary)

n_flagged = bronze_claims.filter(F.col("_bronze_null_flag") == True).count()
print(f"\nRows with _bronze_null_flag=True: {n_flagged:,}")
print("(retained in Bronze for audit trail — excluded in Silver quality gate)")


In [0]:
# Delta time travel — history log
print("=== Delta history: bronze.raw_claims ===")
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.raw_claims LIMIT 5"))


## ✅ Bronze complete

| Table | Rows | Status |
|-------|------|--------|
| `bronze.raw_claims` | ~1M | ✅ Partitioned year/month, ZORDER bene_id |
| `bronze.raw_members` | 10,000 | ✅ Full snapshot with audit columns |

**Design note**: Bronze is append-only. Every raw file ever ingested is preserved and
queryable via Delta time travel — required for HIPAA audit trails and payment accuracy investigation.

**Next**: Run `02_silver_processing` →
